# 00 — Train-only SimCLR pretraining

Pretrain one ConvNeXtV2-Tiny encoder using only the fixed training subjects. The certified fold labels are used once to define a fixed split: folds 2–4 train, fold 1 validation, and fold 0 test. Validation and test image files are not opened in this notebook.

Set `RUN_TRAINING = True` after reviewing the configuration cell. The final checkpoint is consumed by `01_joint_encoder_decoder_training.ipynb`.

In [ ]:
import contextlib
import gc
import json
import math
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms.functional import gaussian_blur

SEED = 42
RUN_TRAINING = False
RESUME_TRAINING = True
MODEL_NAME = "convnextv2_tiny.fcmae_ft_in22k_in1k"
IMAGE_SIZE = 256
BATCH_SIZE = 16
NUM_WORKERS = 4
EPOCHS = 200
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
TEMPERATURE = 0.5
PROJECTION_DIM = 128
USE_AMP = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    print("WARNING: this notebook is intended for the HPC CUDA environment.")
print({"device": str(DEVICE), "run_training": RUN_TRAINING, "epochs": EPOCHS})

In [ ]:
def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "reports" / "manifests" / "quantitative_manifest_v1.csv").is_file():
            return candidate
    raise FileNotFoundError("Could not find the project root from the current working directory.")


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)


ROOT = find_project_root(Path.cwd())
MANIFEST_PATH = ROOT / "reports" / "manifests" / "quantitative_manifest_v1.csv"
OUTPUT_DIR = ROOT / "models" / "joint_simclr_test" / "simclr"
ENCODER_PATH = OUTPUT_DIR / "simclr_encoder.pth"
LAST_CHECKPOINT_PATH = OUTPUT_DIR / "last_checkpoint.pth"
HISTORY_PATH = OUTPUT_DIR / "simclr_history.csv"


def load_fixed_split():
    rows = pd.read_csv(MANIFEST_PATH, dtype={"test_fold": "Int64"})
    rows = rows.loc[rows.status.eq("ready")].copy()
    if len(rows) != 71:
        raise RuntimeError(f"Expected 71 ready knees, found {len(rows)}.")
    rows["test_fold"] = rows.test_fold.astype(int)
    rows["split"] = "train"
    rows.loc[rows.test_fold.eq(1), "split"] = "validation"
    rows.loc[rows.test_fold.eq(0), "split"] = "test"
    expected = {"train": 42, "validation": 14, "test": 15}
    observed = rows.groupby("split").size().to_dict()
    if observed != expected:
        raise RuntimeError(f"Fixed split mismatch: expected {expected}, found {observed}.")
    subjects = {name: set(group.subject_id) for name, group in rows.groupby("split")}
    if subjects["train"] & subjects["validation"] or subjects["train"] & subjects["test"] or subjects["validation"] & subjects["test"]:
        raise RuntimeError("Subject overlap detected between train, validation, and test roles.")
    return {name: rows.loc[rows.split.eq(name)].copy() for name in expected}


splits = load_fixed_split()
split_summary = pd.DataFrame([
    {
        "split": name,
        "knees": len(frame),
        "subjects": frame.subject_id.nunique(),
        "healthy": int(frame.dataset.eq("VSD").sum()),
        "fractured": int(frame.dataset.eq("Ruikar").sum()),
    }
    for name, frame in splits.items()
])
display(split_summary)
seed_everything()

In [ ]:
def read_drr(path: Path) -> np.ndarray:
    array = np.load(path).astype(np.float32)
    if array.shape != (IMAGE_SIZE, IMAGE_SIZE) or not np.isfinite(array).all():
        raise ValueError(f"Invalid DRR: {path}")
    return np.clip(array, 0.0, 1.0)


class SimCLRViewDataset(Dataset):
    """Expose AP and lateral DRRs as separate, training-only SimCLR samples."""
    def __init__(self, rows: pd.DataFrame):
        self.samples = []
        for row in rows.sort_values("sample_id").itertuples(index=False):
            self.samples.append((str(row.sample_id), "ap", ROOT / row.ap_drr_path))
            self.samples.append((str(row.sample_id), "lat", ROOT / row.lat_drr_path))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample_id, view, path = self.samples[index]
        image = torch.from_numpy(read_drr(path)).unsqueeze(0)
        return image, sample_id, view


class GeometryPreservingAugment:
    def __call__(self, image: torch.Tensor) -> torch.Tensor:
        gamma = float(torch.empty(1).uniform_(0.85, 1.15))
        brightness = float(torch.empty(1).uniform_(-0.08, 0.08))
        contrast = float(torch.empty(1).uniform_(0.85, 1.15))
        sigma = float(torch.empty(1).uniform_(0.0, 0.025))
        output = image.clamp(0, 1).pow(gamma)
        mean = output.mean()
        output = (output - mean) * contrast + mean + brightness
        if sigma > 0:
            output = output + torch.randn_like(output) * sigma
        if bool(torch.rand(1) < 0.5):
            output = gaussian_blur(output, kernel_size=[5, 5], sigma=[0.1, 1.5])
        return output.clamp(0, 1).repeat(3, 1, 1)


class SimCLRModel(nn.Module):
    def __init__(self, pretrained: bool):
        super().__init__()
        self.encoder = timm.create_model(MODEL_NAME, pretrained=pretrained, num_classes=0)
        feature_dim = int(self.encoder.num_features)
        self.projector = nn.Sequential(
            nn.Linear(feature_dim, feature_dim),
            nn.ReLU(inplace=True),
            nn.Linear(feature_dim, PROJECTION_DIM),
        )

    def forward(self, image):
        feature = self.encoder(image)
        return feature, F.normalize(self.projector(feature), dim=1)


def nt_xent_loss(first, second, temperature=TEMPERATURE):
    batch = first.shape[0]
    embeddings = F.normalize(torch.cat([first, second], dim=0), dim=1)
    logits = embeddings @ embeddings.T / temperature
    logits.fill_diagonal_(float("-inf"))
    targets = (torch.arange(2 * batch, device=logits.device) + batch) % (2 * batch)
    return F.cross_entropy(logits, targets)


PRETRAINED_CONFIG = timm.get_pretrained_cfg(MODEL_NAME)
NORMALIZATION = {
    "mean": tuple(float(value) for value in PRETRAINED_CONFIG.mean),
    "std": tuple(float(value) for value in PRETRAINED_CONFIG.std),
}
augment = GeometryPreservingAugment()


def normalize_batch(images):
    mean = torch.tensor(NORMALIZATION["mean"], device=images.device, dtype=images.dtype).view(1, 3, 1, 1)
    std = torch.tensor(NORMALIZATION["std"], device=images.device, dtype=images.dtype).view(1, 3, 1, 1)
    return (images - mean) / std


train_dataset = SimCLRViewDataset(splits["train"])
if len(train_dataset) != 84:
    raise RuntimeError(f"Expected 84 training views, found {len(train_dataset)}.")
print({"training_knees": len(splits["train"]), "training_views": len(train_dataset), "normalization": NORMALIZATION})

In [ ]:
def amp_context():
    if not (USE_AMP and DEVICE.type == "cuda"):
        return contextlib.nullcontext()
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return torch.autocast("cuda", dtype=dtype)


def make_scaler():
    enabled = USE_AMP and DEVICE.type == "cuda" and not torch.cuda.is_bf16_supported()
    return torch.amp.GradScaler("cuda", enabled=enabled)


@torch.no_grad()
def similarity_audit(model, loader):
    model.eval()
    positive_values, negative_values, feature_stds = [], [], []
    for images, _, _ in loader:
        first = torch.stack([augment(image) for image in images]).to(DEVICE, non_blocking=True)
        second = torch.stack([augment(image) for image in images]).to(DEVICE, non_blocking=True)
        with amp_context():
            first_features, first_projection = model(normalize_batch(first))
            second_features, second_projection = model(normalize_batch(second))
        positive_values.extend(F.cosine_similarity(first_projection.float(), second_projection.float()).cpu().tolist())
        if len(first_projection) > 1:
            negative_values.extend(F.cosine_similarity(first_projection.float(), second_projection.roll(1, 0).float()).cpu().tolist())
        feature_stds.append(float(torch.cat([first_features, second_features]).float().std(dim=0).mean().cpu()))
    return {
        "positive_similarity": float(np.mean(positive_values)),
        "shuffled_similarity": float(np.mean(negative_values)),
        "feature_std": float(np.mean(feature_stds)),
    }


def save_training_plots(history, audit):
    frame = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(frame.epoch, frame.train_loss)
    axes[0].set(title="SimCLR training loss", xlabel="Epoch", ylabel="NT-Xent loss")
    axes[0].grid(alpha=0.25)
    axes[1].bar(["Positive", "Shuffled"], [audit["positive_similarity"], audit["shuffled_similarity"]])
    axes[1].set(title="Training-view similarity audit", ylabel="Cosine similarity")
    axes[1].grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "simclr_training_summary.png", dpi=180, bbox_inches="tight")
    plt.show()


def train_simclr():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    seed_everything()
    generator = torch.Generator().manual_seed(SEED)
    loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=NUM_WORKERS > 0,
        generator=generator,
    )
    model = SimCLRModel(pretrained=True).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = make_scaler()
    history, start_epoch = [], 0

    if RESUME_TRAINING and LAST_CHECKPOINT_PATH.is_file():
        checkpoint = torch.load(LAST_CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
        model.load_state_dict(checkpoint["model_state"], strict=True)
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        scheduler.load_state_dict(checkpoint["scheduler_state"])
        scaler.load_state_dict(checkpoint["scaler_state"])
        history = checkpoint["history"]
        start_epoch = int(checkpoint["epoch"]) + 1
        print(f"Resuming at epoch {start_epoch}/{EPOCHS}")

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        losses = []
        for images, _, _ in loader:
            first = torch.stack([augment(image) for image in images]).to(DEVICE, non_blocking=True)
            second = torch.stack([augment(image) for image in images]).to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with amp_context():
                _, first_projection = model(normalize_batch(first))
                _, second_projection = model(normalize_batch(second))
                loss = nt_xent_loss(first_projection, second_projection)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite SimCLR loss at epoch {epoch}.")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            losses.append(float(loss.detach().cpu()))
        scheduler.step()
        row = {"epoch": epoch, "train_loss": float(np.mean(losses)), "lr": optimizer.param_groups[0]["lr"]}
        history.append(row)
        pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "scaler_state": scaler.state_dict(),
            "history": history,
        }, LAST_CHECKPOINT_PATH)
        print(f"epoch={epoch + 1:03d}/{EPOCHS} loss={row['train_loss']:.5f} lr={row['lr']:.2e}")

    audit_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    audit = similarity_audit(model, audit_loader)
    if not np.isfinite(list(audit.values())).all() or audit["feature_std"] <= 1e-4:
        raise RuntimeError(f"Collapsed or invalid SimCLR representation: {audit}")
    if audit["positive_similarity"] <= audit["shuffled_similarity"]:
        raise RuntimeError(f"Positive similarity did not exceed shuffled similarity: {audit}")

    torch.save({
        "model_name": MODEL_NAME,
        "encoder_state": model.encoder.state_dict(),
        "normalization": NORMALIZATION,
        "epochs": EPOCHS,
        "similarity_audit": audit,
    }, ENCODER_PATH)
    reload_model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=0)
    reload_model.load_state_dict(torch.load(ENCODER_PATH, map_location="cpu", weights_only=False)["encoder_state"], strict=True)
    save_training_plots(history, audit)
    print({"encoder_checkpoint": str(ENCODER_PATH), **audit})
    del reload_model, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return pd.DataFrame(history), audit


if RUN_TRAINING:
    simclr_history, simclr_audit = train_simclr()
else:
    print("Definitions loaded. Set RUN_TRAINING=True to run train-only SimCLR pretraining.")